# M5 Monthly SKU Demand Forecasting — Debug Notebook

Replica o pipeline de `src/main.py` passo a passo para testar a execução e inspecionar resultados intermediários.

```bash
uv run jupyter notebook notebooks/main.ipynb
```

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from models.neural import NeuralModel
from train.trainer import train_model
from data.reader import load_sales_data, validate_sales_schema
from evaluation.metrics import evaluate
from preprocess.scaler import scale_features, scale_target, unscale_target
from preprocess.transform import (
    MONTH_COLUMN,
    TARGET_AHEAD_COLUMN,
    build_dataset,
    preprocess_series,
)
from utils.pytorch import get_device, make_loader, predict
from preprocess.split import time_based_split
from config import default_config

pd.set_option("display.max_columns", 20)
print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")

## Step 1 — Configuration

In [ ]:
config = default_config()
config

## Step 2 — Load and validate data

In [ ]:
raw_data = load_sales_data(str(config.data_path))
data = validate_sales_schema(raw_data)

print(f"Rows: {len(data):,} | Columns: {list(data.columns)}")
data.head()

## Step 3 — Monthly time series per SKU

In [ ]:
monthly = preprocess_series(data)

print(f"Monthly rows: {len(monthly):,} | SKUs: {monthly['item_id'].nunique()}")
monthly.head(10)

## Step 4 — Build dataset with lag features

In [ ]:
supervised, lag_columns = build_dataset(
    monthly,
    lags=config.lags,
    horizon=config.forecast_horizon,
)

print(f"Dataset rows: {len(supervised):,}")
print(f"Lag columns: {lag_columns}")
supervised.head()

## Step 5 — Time-based split

In [ ]:
print(
    f"Reference forecast month: {config.reference_month} | "
    f"Validation window: {config.validation_months} month(s) | "
    f"Horizon: {config.forecast_horizon}"
)

(X_train, y_train), (X_val, y_val), (X_test, y_test) = time_based_split(
    supervised,
    date_column=MONTH_COLUMN,
    feature_columns=lag_columns,
    target_column=TARGET_AHEAD_COLUMN,
    reference_month=config.reference_month,
    validation_months=config.validation_months,
    forecast_horizon=config.forecast_horizon,
)

for name, X, y in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test),
]:
    print(f"{name:5} X={X.shape}  y={y.shape}  y mean={y.mean():.2f}  y std={y.std():.2f}")

## Step 6 — Scale features and target

In [ ]:
X_train_pp, X_val_pp, X_test_pp = scale_features(
    X_train, X_val, X_test, method="standardize"
)
y_train_s, y_val_s, y_test_s, y_loc, y_scale = scale_target(y_train, y_val, y_test)

print(f"X_train_pp mean={X_train_pp.mean():.4f} std={X_train_pp.std():.4f}")
X_train_pp[:3]

## Step 7 — PyTorch data loaders

In [ ]:
device = get_device()
train_loader = make_loader(
    X_train_pp, y_train_s, batch_size=config.batch_size, shuffle=True
)
val_loader = make_loader(
    X_val_pp, y_val_s, batch_size=config.batch_size, shuffle=False
)

print(f"Device: {device}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## Step 8 — Train, save, and evaluate

In [ ]:
model = NeuralModel(input_dim=X_train_pp.shape[1], hidden_dim=64, output_dim=1)

print("─── Starting Neural Network Training ───")
model = train_model(
    model,
    train_loader,
    val_loader,
    device=device,
    epochs=config.epochs,
    learning_rate=config.learning_rate,
)

In [ ]:
model_path = PROJECT_ROOT / "demand_model.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

In [ ]:
val_predictions = unscale_target(predict(model, X_val_pp, device), y_loc, y_scale)
test_predictions = unscale_target(predict(model, X_test_pp, device), y_loc, y_scale)

val_metrics = evaluate(y_val, val_predictions, split="validation")
test_metrics = evaluate(y_test, test_predictions, split="test")

In [ ]:
print("=" * 60)
print(" Final Summary")
print("=" * 60)
print(f"  Training samples:    {X_train_pp.shape[0]}")
print(f"  Validation samples:  {X_val_pp.shape[0]}")
print(f"  Test samples:        {X_test_pp.shape[0]}")
print(f"  Features (lags):     {X_train_pp.shape[1]}")
print(f"  Validation MAE:      {val_metrics['mae']:.4f}  (MAE/mean={val_metrics['mae_over_mean']:.4f})")
print(f"  Validation RMSE:     {val_metrics['rmse']:.4f}")
print(f"  Test MAE:            {test_metrics['mae']:.4f}  (MAE/mean={test_metrics['mae_over_mean']:.4f})")
print(f"  Test RMSE:           {test_metrics['rmse']:.4f}")
print("=" * 60)
print("\nPipeline finished successfully.")

In [ ]:
# Summary printed in the cell above.